In [102]:
import os
import music21 as m21
import numpy as np
import pandas as pd
import json
import tensorflow.keras as keras

KERN_DATASET_PATH=r'C:\Users\sresh\OneDrive\Documents\code\ML\music_gen_LSTM\ESAC_music_dataset\essen\europa\deutschl\erk'

ACCEPTABLE_DURATIONS_LIST=[
    0.25,                                  # quarter note, it is the step in our time series data
    0.5,
    0.75,                                  # dotted note (eighth note + sixteenth note)
    1.0,
    1.5,
    2,
    3,
    4                                      # full note
   
]

SAVE_DIR=r"C:\Users\sresh\OneDrive\Documents\code\ML\music_gen_LSTM\dataset"

SINGLE_FILE_DATASET=r"C:\Users\sresh\OneDrive\Documents\code\ML\music_gen_LSTM\single_file_dataset"

SEQUENCE_LENGTH= 64         # length of sequence to be used in LSTM

MAPPING_PATH=r"C:\Users\sresh\OneDrive\Documents\code\ML\music_gen_LSTM\mapping.json"

# Creating the Music21 environment to run with MuseScore 4

In [103]:
env=m21.environment.Environment()
env['musicxmlPath'] = r'C:\Program Files\MuseScore 4\bin\MuseScore4.exe'
env['musescoreDirectPNGPath'] = r'C:\Program Files\MuseScore 4\bin\MuseScore4.exe'
env['musicxmlPath']

WindowsPath('C:/Program Files/MuseScore 4/bin/MuseScore4.exe')

# Procedure

- Load dataset using Music21

- Filter out songs with non-acceptable durations 

- Transpose songs to Cmaj/Amin 

- Encode songs with music time series representation

- Save songs to a text file

# Loading songs in Kern

In [104]:
def load_songs_in_kern(dataset_path):

    songs=[]
    # go through all files in the dataset and load them with Music21
    for path,subdir,files in os.walk(dataset_path):
        for file in files:
            if file[-3:]=="krn":
                song=m21.converter.parse(os.path.join(path,file))
                songs.append(song)
    
    return songs

# Checking if songs have acceptable durations

In [105]:
 def has_acceptable_duration(song,acceptable_durations):
        for note in song.flat.notesAndRests:
            if note.duration.quarterLength not in acceptable_durations:
                return False
        
        return True

# Transposing songs to 2 keys, Cmaj and Amin

In [106]:
def transpose(song):
    # Get the first part from the song
    part = song.parts[0]                               # we want to work with the first part

   
    # We want to extract the key signature from the first measure of the first part
    measures_part0 = part.getElementsByClass(m21.stream.Measure)
    if measures_part0:
        first_measure = measures_part0[0]
        key = first_measure.getElementsByClass(m21.key.Key)
        if key:
            key = key[0]
        else:
            # If key is not found in the measure, estimate it
            key = song.analyze("key")
    else:
        # Handle the case where there are no measures in the first part
        key = song.analyze("key")

    print(key)
    

    # Calculate the transposition interval based on the key
    if key.mode == "major":
        interval = m21.interval.Interval(key.tonic, m21.pitch.Pitch("C"))
    elif key.mode == "minor":
        interval = m21.interval.Interval(key.tonic, m21.pitch.Pitch("A"))
        

    # Transpose the song by the calculated interval
    transposed_song = song.transpose(interval)

    return transposed_song


# Encoding notes and rests

In [107]:
def encode_song(song, time_step=0.25):
    
    # Eg. p=60 (MIDI note for the pitch) , d=1.0 (duration) -->  list [60,"_","_","_"]
    
    encoded_song=[]
    
    for event in song.flat.notesAndRests:
        
        # handling notes
        if isinstance(event,m21.note.Note):
            symbol=event.pitch.midi           # 60
            
        #handling rests
        elif isinstance(event, m21.note.Rest):
            symbol="r"
            
        # convert notes and rests into time series
        steps=int(event.duration.quarterLength/time_step)
 
        for step in range(steps):
            if step==0:
                encoded_song.append(symbol)
            else:
                encoded_song.append("_")
    
    # cast encoded song to a string
    
    encoded_song=" ".join(map(str,encoded_song))
    
    return encoded_song

# Preprocessing

In [108]:
def preprocess(dataset_path):
    
    print("Loading songs......")
    songs=load_songs_in_kern(dataset_path)
    print(f"Loaded length of songs is {len(songs)}")
    
    # filter out songs with non-acceptable lengths
    
    for i,song in enumerate(songs):
        if not has_acceptable_duration(song,ACCEPTABLE_DURATIONS_LIST):
            continue
    
    
        # transpose songs to Cmaj/Amin
        song=transpose(song)


        # encode song with music time series representation
        encoded_song=encode_song(song)

        # save songs to a text file
        save_path=os.path.join(SAVE_DIR,str(i))
        with open(save_path,"w") as fp:
            fp.write(encoded_song)


# Concatenating different encoded files into one dataset file for the neural network

Define the load function

In [109]:
def load(file_path):
    with open(file_path,"r") as fp:
        song=fp.read()
    return song

In [110]:
def create_single_file_dataset(dataset_path, file_dataset_path, sequence_length):
    
    new_song_delimiter="/ " * sequence_length    # number of delimiters should match the sequence length for the LSTM
    
    songs=""
    
    # load encoded songs and add delimiters
    for path,_,files in os.walk(dataset_path):
        for file in files:
            file_path = os.path.join(path,file)
            song=load(file_path)
            songs=songs+song+" "+new_song_delimiter
            
    songs=songs[:-1]          # remove empty space from end of song
    
    
    # save string that contains the dataset
    
    with open(file_dataset_path,"w") as fp:
        fp.write(songs)
        
    return songs

# Mapping symbols (like '_' and 'r') to integers for the neural network

In [111]:
def create_mapping(songs,mapping_path):
    
    mappings={}
    
    # identify vocabulary 
    songs=songs.split()                                  # creating list with all symbols and integers in the dataset
    vocabulary=list(set(songs))                          # set() reduces the dataset to just unique symbols
    
    
    # create mappings
    for i,symbol in enumerate(vocabulary):
        mappings[symbol]=i
    
    
    # save vocabulary to a json file
    with open(mapping_path,"w") as fp:
        json.dump(mappings,fp, indent=4)
    

# Converting songs to integers

In [112]:
def convert_songs_to_int(songs):
    
    int_songs=[]
    
    # load the mappings
    with open(MAPPING_PATH,"r") as fp:
        mappings=json.load(fp)
    
    
    # cast songs string to a list
    songs=songs.split()
    
    
    # map songs to int
    for symbol in songs:
        int_songs.append(mappings[symbol])
        
    return int_songs

# Generating training examples

In [113]:
def generate_training_sequences(sequence_length):

    
    # [11,12,13,14,.....]  -->  create sequences,say,of length 2 --> i:[11,12] and t:13 (target)
    # keep sliding i to the right, and the next value is the target (predicted time series output)
    
    
    
    # load songs and map them to int
    songs=load(SINGLE_FILE_DATASET)
    int_songs=convert_songs_to_int(songs)
    
    
    
    
    # generate training sequences
    # if 100 symbols and 64 sequence length, then 100-64=36= number of training sequences
    inputs=[]
    targets=[]
    num_sequences=len(int_songs)-sequence_length
    
    for i in range(num_sequences):
        inputs.append(int_songs[i:i+sequence_length])           # adding sliding windows of time series to the inputs and targets
        targets.append(int_songs[i+sequence_length])
        
    
    
    
    # one-hot encoding of sequences
    
    # inputs : (number of sequences, sequence_length)
    # eg.  [[0,1,2],[1,1,2]]  -->  [  [[1,0,0],[0,1,0],[0,0,1]],   [[0,1,0],[0,1,0],[0,0,1]]   ]    (adding extra dimension)
    
    vocabulary_size=len(set(int_songs))
    inputs=keras.utils.to_categorical(inputs, num_classes=vocabulary_size)
    targets-np.array(targets)
    
    return inputs,targets
    

# main 

In [114]:
if __name__== "__main__":
    
#     songs=load_songs_in_kern(KERN_DATASET_PATH)
#     print(f"Loaded {len(songs)} songs")
#     song=songs[0]
    
#     print(f"Has acceptable duration? {has_acceptable_duration(song,ACCEPTABLE_DURATIONS)}")
    
    preprocess(KERN_DATASET_PATH)
    
    songs=create_single_file_dataset(SAVE_DIR,SINGLE_FILE_DATASET,SEQUENCE_LENGTH)
    create_mapping(songs,MAPPING_PATH)
    transposed_song=transpose(song)
    
    
    inputs,targets=generate_training_sequences(SEQUENCE_LENGTH)
    
    print(inputs.shape)
    print(len(targets))
    
#     song.show()
#     transposed_song.show()
    
    song.write('musicxml', 'output_file_non_transposed.musicxml')
    transposed_song.write('musicxml', 'output_file_transposed.musicxml')

    


Loading songs......
Loaded length of songs is 1700
F major
F major
F major
F major
F major
G major
G major
A major
A major
F major
D major
G major
E- major
F major
G major
B- major
C major
G major
F major
C major
G major
A major
A major
G major
G major
G major
G major
a minor
G major
B- major
G major
G major
F major
E- major
B- major
B- major
F major
G major
C major
F major
F major
a minor
a minor
f minor
e minor
G major
G major
B- major
G major
G major
G major
G major
G major
G major
G major
F major
E- major
E- major
E- major
F major
C major
C major
C major
G major
F major
e minor
G major
A major
F major
F major
G major
G major
G major
G major
F major
G major
G major
G major
G major
G major
B- major
G major
G major
E- major
A major
G major
G major
G major
G major
f minor
G major
G major
G major
G major
G major
G major
g minor
G major
F major
D major
F major
F major
g minor
F major
F major
G major
D major
D major
D major
E- major
G major
G major
g minor
F major
F major
G major
G major


G major
F major
G major
G major
G major
F major
G major
F major
G major
G major
C major
C major
D major
F major
G major
C major
F major
G major
F major
F major
F major
F major
G major
G major
G major
C major
G major
A major
G major
G major
C major
C major
C major
D major
G major
G major
g minor
G major
F major
G major
G major
G major
G major
C major
F major
G major
G major
G major
G major
F major
G major
G major
C major
C major
C major
F major
A major
C major
F major
G major
D major
G major
G major
F major
G major
g minor
G major
G major
G major
A major
F major
G major
D major
G major
G major
G major
C major
G major
G major
F major
C major
F major
g minor
C major
B- major
G major
G major
G major
F major
C major
C major
F major
F major
F major
F major
F major
F major
F major
F major
F major
C major
G major
F major
G major
G major
F major
F major
F major
G major
G major
F major
C major
e minor
F major
F major
g minor
C major
G major
G major
G major
G major
G major
G major
g minor
G major

# Model 

### (using functional Keras API instead of Sequential approach)

In [119]:
OUTPUT_UNITS=38    # size of vocabulary 

LOSS="sparse_categorical_crossentropy"

LEARNING_RATE=0.001

NUM_UNITS=[256]

EPOCHS=50

BATCH_SIZE=64

SAVE_MODEL_PATH="model.h5"

In [120]:
def build_model(output_units,num_units,loss,learning_rate):
    
    # create model architecture
    input=keras.layers.Input(shape=(None,output_units))
    x= keras.layers.LSTM(num_units[0])(input)             # passing input to LSTM layer
    x=keras.layers.Dropout(0.2)(x)                        # Dropout to avoid overfitting
    
    output=keras.layers.Dense(output_units,activation="softmax")(x)
    
    model=keras.Model(input,output)
    
    # compile model
    
    model.compile(loss=loss,
                  optimizer=keras.optimizers.Adam(lr=learning_rate),
                  metrics=["accuracy"]
                 )
    
    model.summary()
    
    return model


In [126]:
def train(output_units=OUTPUT_UNITS,num_units=NUM_UNITS,loss=LOSS,learning_rate=LEARNING_RATE):
    
    # generate training sequences
    inputs,targets=generate_training_sequences(SEQUENCE_LENGTH)
    
    # build the network
    model=build_model(output_units,num_units,loss,learning_rate)
    
    # train the model
    inputs = np.array(inputs)
    targets = np.array(targets)

    model.fit(inputs,targets,epochs=EPOCHS, batch_size=BATCH_SIZE)
    
    # save the model
    model.save(SAVE_MODEL_PATH)
    

In [127]:
if __name__=="__main__":
    train()

Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, None, 38)]        0         
                                                                 
 lstm_3 (LSTM)               (None, 256)               302080    
                                                                 
 dropout_3 (Dropout)         (None, 256)               0         
                                                                 
 dense_3 (Dense)             (None, 38)                9766      
                                                                 
Total params: 311846 (1.19 MB)
Trainable params: 311846 (1.19 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/50
5663/5663 [==============================] - 513s 90ms/step - loss: 0.6816 - accuracy: 0.7895
Epoch 2/50
5663/5663 [============================

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)



5663/5663 [==============================] - 3980s 703ms/step - loss: 0.2427 - accuracy: 0.9166
Epoch 49/50
5663/5663 [==============================] - 544s 96ms/step - loss: 0.2428 - accuracy: 0.9167
Epoch 50/50
5663/5663 [==============================] - 565s 100ms/step - loss: 0.2380 - accuracy: 0.9180


C:\Users\sresh\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
